# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring a dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library, referencing dataset fields and elements by their Croissant `@id`s for clear, standards-based data access.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We access the dataset with its Croissant schema, and print key metadata fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset (instantiates a Dataset object)
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata using the 'to_json' method
metadata = dataset.metadata.to_json()

print(f"Name: {metadata.get('name')}")
print(f"Description: {metadata.get('description')}")
print(f"License: {metadata.get('license')}")
print(f"Authors (@id): {[a.get('@id') for a in metadata.get('author', [])]}")
print(f"Keywords: {metadata.get('keywords')}")
print(f"Spatial coverage: {metadata.get('spatialCoverage')}")
print(f"Temporal coverage: {metadata.get('temporalCoverage')}")
print("\nFull metadata example snippet:")
pprint({k: metadata[k] for k in list(metadata)[:6]})

## 2. Data Overview
Review available record sets, fields, and their IDs (using Croissant `@id`).

We list the record sets defined in the schema. Each record set is referenced via its `@id`, which is used for programmatic access.

If your dataset does not expose record sets automatically, you can list their `@id`s from the metadata.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s). Their @id's:")
for rs in record_sets:
    print(f"- Record set @id: {rs['@id']}")
    # Print summary of the fields in each record set
    fields = rs.get('field', [])
    # field can be either a list of dicts or a single dict
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (@id):")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id')} (name: {f.get('name')})")
        else:
            print(f"    - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record sets, fields, and columns are referenced by their `@id`.

We extract data from each record set defined in the dataset, using their `@id`, and load all records into pandas DataFrames for further processing.

In [ ]:
# Build a mapping of DataFrames for each record set by @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set '@id': {record_set_id} [{df.shape[0]} rows, {df.shape[1]} columns]")
        print(f"Columns (@id): {list(df.columns)}\n")
    else:
        print(f"No records found for record set '@id': {record_set_id}\n")

# For demonstration: Pick the first non-empty record set
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f"Example columns for record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter by criteria, normalize numeric fields, and group data.

**Always use the Croissant `@id` to reference dataset fields or columns.**

> For demonstration, we select the first numeric field (`@id`) in the main record set, filter for records with values above a chosen threshold, normalize this column, and group by a categorical field if available.

In [ ]:
import numpy as np

# Identify suitable fields for numeric analysis
df = dataframes[main_record_set_id]
# Try to infer numeric columns (as Croissant @id)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # use first numeric @id
    print(f"Numeric field selected for EDA (by @id): {numeric_field_id}")
    
    threshold = np.percentile(df[numeric_field_id].dropna(), 80)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 20%): {len(filtered_df)} record(s)")
    display(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try to find a grouping field (non-numeric, likely categorical)
    cat_fields = [col for col in df.columns if df[col].dtype == 'object']
    group_field_id = cat_fields[0] if cat_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}: (Display mean of {numeric_field_id})")
        display(grouped_df.head())
else:
    print("No numeric field found for analysis in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields using referenced `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No numeric field found.")

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore a Croissant-structured dataset programmatically via the `mlcroissant` library. Key steps included:

- Accessing all entities (record sets, fields) strictly via their Croissant `@id`.
- Dynamically loading metadata and tabular data into pandas for analysis.
- Executing basic exploratory data analysis and visualizations.

Continue your analysis by referencing the Croissant schema for additional field semantics or by combining fields using their `@id` for downstream tasks like modeling or policy evaluation.